In [5]:
%load_ext autoreload
%autoreload 2

import os,sys
import torch
import numpy as np
import pandas as pd 

from torch import nn
from torchdiffeq import odeint

from PINN import models
import matplotlib.pyplot as plt

import umap
import phate

from scipy.stats import gaussian_kde

os.chdir("/ssd/users/Wergillius/Project/PINN_dynamics")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# reload synthesized data

In [2]:
syn_init_ckpt = torch.load("data/synthesized_data/5Dim_ncs_synparam_Jan21/syn_init_condition.ckpt")
cellstate = syn_init_ckpt['cellstate']

In [6]:
syn_result_ckpt = torch.load("data/synthesized_data/5Dim_ncs_synparam_Jan21/syn_result.ckpt")
density = syn_result_ckpt['u_simulate']
integrate_time = syn_result_ckpt['integrate_time']
delta_X = syn_result_ckpt['delta_x']

# load dataloader

In [9]:
from torch.utils.data import Dataset, DataLoader

In [4]:
from PINN import reader

In [12]:
seen_timepoints = [0, 2, 4, 6, 8, 10]
leaveout_timepoints = [1,3,7]
test_timepoitns = [5,9]

train_DS = reader.Syn_DS(cellstate=cellstate, 
                  density=density[seen_timepoints],
                  integrate_time= integrate_time[seen_timepoints],
                  deltax=delta_X,
                  batchsize = 64
                  )

val_DS = reader.Syn_DS(cellstate=cellstate, 
                  density=density[leaveout_timepoints],
                  integrate_time= integrate_time[leaveout_timepoints],
                  deltax=delta_X,
                  batchsize = 512)

train_DL = DataLoader(train_DS, batch_size=None, num_workers=5)
val_DL = DataLoader(val_DS, batch_size=None, num_workers=5)

In [18]:
dataset_kws = dict(
    seen_timepoints = [0, 2, 4, 6, 8, 10],
    leaveout_timepoints = [1,3,7],
    test_timepoitns = [5,9],
)

# define model

In [16]:
model_kws = dict(
    channels =   [5+1, 16, 16, 1],
    g_channels = [5, 16, 16, 1],
    v_channels = [5, 32, 16, 5],
    D_channels = [5, 32, 1],
    activation_fn='Tanh',
    ode_tol = 1e-4,
    D_penalty = 0.5,
    deltax_weight = 3,
    weight_intensity = 1,
    time_scale_factor = 1,
    time_sensitive = False
)
model = models.pde_params(**model_kws)

In [19]:
config = model_kws
config.update(dataset_kws)

In [21]:
config

{'channels': [6, 16, 16, 1],
 'g_channels': [5, 16, 16, 1],
 'v_channels': [5, 32, 16, 5],
 'D_channels': [5, 32, 1],
 'activation_fn': 'Tanh',
 'ode_tol': 0.0001,
 'D_penalty': 0.5,
 'deltax_weight': 3,
 'weight_intensity': 1,
 'time_scale_factor': 1,
 'time_sensitive': False,
 'seen_timepoints': [0, 2, 4, 6, 8, 10],
 'leaveout_timepoints': [1, 3, 7],
 'test_timepoitns': [5, 9]}

In [23]:
import yaml

config = model_kws
config.update(dataset_kws)

with open("logs/5Dim_ncs_synparam_Jan21/v1_Jan22.yaml", 'w') as outfile:
    yaml.dump(config, outfile, default_flow_style=False)

In [22]:
import pytorch_lightning as pl

In [24]:
trainer = pl.Trainer(
                    auto_lr_find=True,
                    accelerator='gpu',
                    default_root_dir="logs/5Dim_ncs_synparam_Jan21",
                    devices = [7], 
                    max_epochs=300,
                    callbacks=[pl.callbacks.ModelCheckpoint(
                                filename='{epoch}-{val_loss:.8f}',
                                monitor="total_loss", mode="min", save_top_k=2)]
                    )

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [25]:
trainer.fit(model, train_dataloaders=train_DL, val_dataloaders=val_DL)

You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]

  | Name    | Type            | Params
--------------------------------------------
0 | GNLL_fn | GaussianNLLLoss | 0     
1 | KLD_fn  | KLDivLoss       | 0     
2 | u       | MLP_surrogate   | 401   
3 | g       | MLP_surrogate   | 385   
4 | v       | MLP_surrogate   | 805   
5 | D       | MLP_surrogate   | 225   
--------------------------------------------
1.8 K     Trainable params
0         Non-trainable params
1.8 K     Total params
0.007     Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

/ssd/users/Wergillius/miniforge3/envs/PINN_torch/lib/python3.9/site-packages/pytorch_lightning/trainer/call.py:48: UserWarning: Detected KeyboardInterrupt, attempting graceful shutdown...
  rank_zero_warn("Detected KeyboardInterrupt, attempting graceful shutdown...")
